# Practice 03 - conditional_edge
## 상담봇
### State
```
user_input: str  # 사용자 메세지
sentiment: str (positive, negative, agreesive)
core_msg: str
warning_count: int  # 상담할때, 공격적인 언행을 하면 경고 누적 횟수
response: str  # 답변
```
### Node
- `block_user`: 제공
- `analyze_sentiment`: LLM이 `user_input` 을 분석하여 `sentiment`를 `['positive', 'negative', 'agreesive']` 중 한가지로만 세팅
- `positive_node`: 제공
- `negative_node`: 제공
- `aggressive_node`: 제공
- `make_final_msg`: LLM이 `core_msg` 를 보고 최종 답변 생성

### Router
- `block_router` : `state['warning_count']`를 확인(3이상인지)하고 다음 노드 결정
- `emotion_router`: 단순히 `state['sentiment']`를 보고 다음 노드 결정

### Flow
```mermaid
flowchart TD
    START([START]) --> block_router{block_router}
    
    %% START 분기
    block_router -->|gogo| analyze_sentiment[analyze_sentiment]
    block_router -->|block| block_user[block_user]
    
    %% block_user 종료
    block_user --> END([END])
    
    %% 감정 분석 후 분기
    analyze_sentiment --> emotion_router{emotion_router}
    emotion_router -->|positive| positive_node[positive_node]
    emotion_router -->|negative| negative_node[negative_node]
    emotion_router -->|aggressive| aggressive_node[aggressive_node]
    
    %% 최종 메시지 생성으로 모임
    positive_node --> make_final_msg[make_final_msg]
    negative_node --> make_final_msg[make_final_msg]
    aggressive_node --> make_final_msg[make_final_msg]
    
    %% 최종 메시지 종료
    make_final_msg --> END
```

In [75]:
import random
from typing import TypedDict, Literal ##객관식으로 다음에 오는 것들 중 하나로 답변되게 하는법
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

load_dotenv()


True

In [76]:
#State
class ChatState(TypedDict): 
    user_input: str  #사용자 메세지
    sentiment: Literal['positive', 'negative', 'agreesive'] ## 반드시 3개중 하나만 값이 들어가게함
    core_msg: str
    warning_count: int  #상담할때, 공격적인 언행을 하면 경고 누적 횟수
    response: str  #답변

# class Emotion(BaseModel):
#     emotion : str = Field(description='''입력 받은 메세지에서 감정을 분석해서
#     1. positive
#     2. negative
#     3. aggresive
#     위의 세가지 값중 하나만 출력해라 
#        ''')

In [77]:
#
def analyze_sentiment(state: ChatState):
    llm = init_chat_model('openai:gpt-4.1-mini')
    # structured_llm = llm.with_structured_output(Emotion)

    result = llm.invoke([
        {'role':'system','content': '''너는 입력받은 메세지에서 느껴지는 감정상태를 분석해주는 AI야.
        입력 받은 메세지를 바탕으로 감정을 positive, negative, aggresive 하나로 반환해줘
        '''},
        {'role': 'user', 'content': state['user_input']}
    ])

    return{'sentiment': result.content}

#
def positive_node(state: ChatState):
    msgs = ['최고', '멋져', '훌륭']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}

#
def negative_node(state: ChatState):
    msgs = ['힘내', '위로', '괜찮']
    keyword = random.choice(msgs)
    return {'core_msg': keyword}

#
def aggressive_node(state: ChatState):
    # dict.get(a, b)  -> key a가 있으면, 해당 value. key a가 없으면 b가 나옴
    count = state.get('warning_count', 0) + 1  # state에 'warning_count' 키가 있으면, 그대로 사용. 없으면 0
    # if state['sentiment'] == 'aggresive':
    #     count += 1
    return {'core_msg':  '공격적인 표현은 삼가라', 'warning_count': count}

#
def make_final_msg(state : ChatState):
    llm = init_chat_model('openai:gpt-4.1-mini')
    result = llm.invoke([
        {'role':'system','content':'너는 상담을 해주는 Ai야 사용자 입력 메세지에대한 답변을 입력받은 단어를 포함해서 반환해줘'},
        {'role':'user','content':f'''core_msg : {state['core_msg']}
사용자 입력 메세지 : {state['user_input']}
'''}
])
    return{'response':result.content}

#
def block_user(state: ChatState):
    return {'response': '님 차단'}

In [ ]:
# 유저차단 확인 라우터
def block_router(state:ChatState):
  if state.get('warning_count', 0) >= 3: #Dict에서 Get메서드를 사용하면 조회값이 없을때도 오류없이 이용 할 수 있다. 없으면 0이 반환되게 메서드로 설정했기 때문에
    return 'Blocked'
  else:
    return 'OK'
  
# 유저 메세지 감정 확인 라우터
def emotion_router(state:ChatState):
  # if state['sentiment'] == 'positive':
  #   return 'positive'
  # elif state['sentiment'] == 'negative':
  #   return 'negative'
  # else:
  #   return 'aggresive'
  return state['sentiment'] ## 이거랑 위의 주석처리된거랑 똑같다!! 이걸 생각 못했네..

In [90]:
builder = StateGraph(ChatState)

builder.add_node('감정분석노드', analyze_sentiment)
builder.add_node('긍정노드',positive_node)
builder.add_node('부정노드',negative_node)
builder.add_node('공격적노드',aggressive_node)
builder.add_node('답변노드',make_final_msg)
builder.add_node('유저차단노드',block_user)

builder.add_conditional_edges(START,block_router,{'OK':'감정분석노드','Blocked':'유저차단노드'})
builder.add_edge('유저차단노드', END)

builder.add_conditional_edges('감정분석노드',emotion_router,{'aggresive':'긍정노드','negative':'부정노드','aggresive':'공격적노드'})
builder.add_edge('긍정노드','답변노드')
builder.add_edge('부정노드','답변노드')
builder.add_edge('공격적노드','답변노드')
builder.add_edge('답변노드',END) ## END로 끝내지 않아도 더이상 노드가 없으면 끝난것으로 인식함 여기서는 그냥 명시적으로 보여주기위해서 작성

graph = builder.compile()

In [91]:
graph.invoke({'user_input':'너 죽고싶냐?','warning_count':1})

{'user_input': '너 죽고싶냐?',
 'sentiment': 'aggresive',
 'core_msg': '공격적인 표현은 삼가라',
 'warning_count': 2,
 'response': '공격적인 표현은 삼가라. 너 죽고싶냐?라는 말은 상처를 줄 수 있으니 조심하는 것이 좋아요.'}